
# 레슨 03 — pandas Series · DataFrame 기초

NumPy 는 빠른 숫자 배열을 다루는 도구였다. pandas 는 여기에 **행 이름과 열 이름**을 붙여 현실의 표 데이터를 다루게 해 주는 도구다. 이번 레슨부터는 학생이 실무 CSV 를 열었을 때 “어떤 열이 있고, 몇 행이 있고, 숫자는 어떻게 요약되는가”를 스스로 확인하는 흐름을 연습한다.

## 학습 목표

이 레슨을 마치면 다음을 할 수 있다.

1. `Series` 와 `DataFrame` 의 차이를 설명한다.
2. CSV 파일을 `pd.read_csv` 로 불러오고 `shape`, `columns`, `dtypes` 로 구조를 확인한다.
3. `head`, `tail`, `info`, `describe` 로 데이터 첫 점검을 수행한다.
4. 열 선택, 행 선택, 조건 선택을 `[]`, `.loc`, `.iloc` 로 구분해 사용한다.
5. 계산 열 `revenue` 를 만들고 매출 합계, 평균 주문금액, 상위 주문을 구한다.
6. 범주형 열에 `value_counts` 를 적용해 분포를 요약한다.
7. 출력 숫자를 보고 “어떤 채널/카테고리가 중요한가”를 한 문장으로 쓴다.

---

## 1. 왜 pandas 를 배우는가

현실 데이터는 보통 숫자만 있는 배열이 아니다. 날짜, 지역, 상품 코드, 고객 등급, 수량, 가격처럼 서로 다른 종류의 값이 한 표에 섞여 있다. NumPy 배열로도 다룰 수 있지만, 열 이름이 없으면 `raw[:, 6]` 이 가격인지 할인율인지 계속 기억해야 한다. pandas 는 이 문제를 해결한다.

| 도구 | 잘하는 일 | 약한 점 |
|---|---|---|
| NumPy | 같은 타입 숫자를 빠르게 계산 | 열 이름과 문자열 처리에 약함 |
| pandas | 표 형태 데이터 탐색, 정제, 집계 | 내부는 NumPy보다 무거울 수 있음 |
| Excel | 눈으로 확인하고 간단히 편집 | 반복 자동화와 대용량 처리에 약함 |

pandas 의 핵심은 `DataFrame` 이다. 한 줄로 말하면 **열 이름이 있는 2차원 표**다. 열 하나만 꺼내면 `Series` 가 된다. Series 는 이름 붙은 1차원 배열, DataFrame 은 Series 여러 개를 옆으로 붙인 표라고 보면 된다.

> **🥄 수학 지식 한스푼 — 관측치(observation)와 변수(variable)**
>
> - **뜻**: 관측치는 표의 한 행, 변수는 표의 한 열이다.
> - **수식**: 데이터 표를 `n × p` 로 쓰면 `n` 은 행 수(관측치 수), `p` 는 열 수(변수 수)다.
> - **읽는 법**: `shape == (1000, 9)` 라면 관측치 1000개, 변수 9개라는 뜻이다.
> - **예시**: 주문 데이터에서 주문 1건은 관측치, `quantity` 와 `unit_price` 는 변수다.

현업에서는 “행이 무엇을 의미하는가?”를 먼저 묻는다. 같은 1000행이라도 주문 1000건인지, 고객 1000명인지, 상품 1000개인지에 따라 분석 질문이 완전히 달라진다. 이번 데이터는 **주문 1건 = 행 1개** 이므로 고객 한 명이 여러 번 주문했다면 여러 행으로 나타날 수 있다.

### 1-1. pandas 의 역사 한 줄

pandas 는 2008년 Wes McKinney 가 금융 시계열 분석을 빠르게 하려고 만들었다. 이름은 “panel data” 에서 왔지만, 지금은 금융을 넘어 공공데이터, 로그 분석, 대회 EDA 의 표준 도구가 되었다. 캐글 노트북에서 `import pandas as pd` 를 거의 항상 보는 이유가 여기에 있다.

---

## 2. 환경 셀과 데이터 코드표

이번 레슨의 데이터는 `sales_orders.csv` 하나다. 파일 크기를 50KB 이하로 유지하기 위해 일부 값은 코드로 저장했다.

In [ ]:
import os
import pandas as pd

IS_COLAB = "COLAB_GPU" in os.environ or "COLAB_TPU_ADDR" in os.environ
DATA_BASE = "https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-data-analysis/lectures/03/data" if IS_COLAB else "./data"
print("data base:", DATA_BASE)
print("pandas:", pd.__version__)

| 컬럼 | 의미 | 코드 설명 |
|---|---|---|
| `order_id` | 주문 번호 | 1부터 시작 |
| `date` | 주문일 | 2025년 1~6월 |
| `region` | 지역 | N=북부, S=남부, E=동부, W=서부, C=중앙 |
| `channel` | 판매 채널 | W=웹, A=앱, S=매장 |
| `category` | 상품군 | BK=도서, GD=기기, FD=식품, ST=문구, TY=완구 |
| `quantity` | 수량 | 1~5개 |
| `unit_price` | 단가 | 원 |
| `discount_rate` | 할인율 | 0~0.2 |
| `customer_grade` | 고객 등급 | N=신규, R=일반, V=VIP |

---

## 3. CSV 불러오기와 첫 점검

데이터 분석의 첫 셀은 멋진 모델이 아니라 “파일을 제대로 읽었는가” 확인이다.

In [ ]:
df = pd.read_csv(f"{DATA_BASE}/sales_orders.csv")
print(type(df))
print("shape:", df.shape)
print("columns:", list(df.columns))

`shape` 는 NumPy 와 똑같이 `(행, 열)` 이다. pandas 는 여기에 `columns` 와 `index` 를 추가로 가진다.

In [ ]:
print(df.head())
print(df.tail(3))

`head()` 는 앞 5행, `tail()` 은 뒤 5행을 보여준다. 괄호 안에 숫자를 넣으면 그 개수만큼 본다. 처음과 끝을 같이 보는 이유는 파일이 중간에 잘렸는지, 마지막 줄에 이상한 합계 행이 붙었는지 빠르게 확인하기 위해서다.

In [ ]:
print(df.info())

`info()` 는 행 수, 열 이름, 결측치가 아닌 값 개수, 자료형을 보여준다. 숫자처럼 보이는 열이 `object` 로 읽혔다면 정제 대상이다. 이번 데이터는 정제 레슨 전이라 결측치가 없고 타입도 정상이다.

In [ ]:
print(df.describe())

`describe()` 는 숫자 열의 `count`, `mean`, `std`, `min`, `25%`, `50%`, `75%`, `max` 를 한 번에 보여준다. 레슨 01에서 NumPy로 하나씩 구했던 요약 통계가 pandas 에서는 표로 나온다.

---

## 4. Series 와 DataFrame

열 하나를 대괄호로 꺼내면 `Series` 다. 여러 열을 리스트로 꺼내면 `DataFrame` 이다.

In [ ]:
# ✏️ 여기에 코드를 작성하세요



`df["quantity"]` 와 `df[["quantity"]]` 는 다르다. 전자는 Series, 후자는 열 하나짜리 DataFrame 이다. 실무에서 함수가 Series 를 기대하는지 DataFrame 을 기대하는지에 따라 결과가 달라질 수 있다.

In [ ]:
print("수량 평균:", qty.mean())
print("수량 최댓값:", qty.max())
print("카테고리 종류:", df["category"].unique())

`Series` 는 NumPy 배열처럼 평균과 최댓값을 바로 계산할 수 있다. 문자열 열에는 `unique()` 로 고유값을 확인한다.

### 4-1. index 는 숨은 이름표다

pandas 객체에는 항상 `index` 가 있다. 지금은 0부터 시작하는 기본 번호라 눈에 잘 띄지 않지만, 날짜나 학생번호를 index 로 쓰는 순간 훨씬 중요해진다.

In [ ]:
print(df.index)
print(df.loc[0, "category"])

`df.loc[0, "category"]` 는 “index 라벨 0번 행의 category 열”을 뜻한다. 현재는 `iloc[0]` 과 결과가 비슷하지만, 나중에 날짜 index 를 쓰면 `.loc["2025-01-01"]` 처럼 날짜로 행을 찾을 수 있다. 09번 시계열 레슨에서 이 차이가 크게 중요해진다.

### 4-2. object dtype 을 조심하자

`region`, `channel`, `category` 같은 문자열 열은 pandas 에서 보통 `object` 로 보인다. 초보자는 `object` 라는 말 때문에 “아무거나 들어 있는 이상한 타입”으로 느끼지만, 처음에는 **문자열 또는 범주형 코드가 들어 있는 열** 정도로 이해하면 된다.

In [ ]:
print(df.dtypes)
print(df["category"].dtype)

정제 레슨에서는 숫자처럼 생긴 문자열, 날짜처럼 생긴 문자열을 실제 숫자와 날짜로 바꾸는 작업을 한다. 이번 데이터는 이미 읽기 좋은 형태라 구조 이해에 집중한다.

---

## 5. 행 선택: `.iloc` 와 `.loc`

pandas 에는 행을 고르는 방법이 많다. 초반에는 두 가지만 구분하면 충분하다.

- `.iloc`: 정수 위치 기반. “0번째 행, 3번째 열”처럼 위치로 고른다.
- `.loc`: 라벨 기반. “인덱스 이름, 열 이름”으로 고른다.

In [ ]:
print(df.iloc[0])        # 첫 번째 행 전체
print(df.iloc[:3, :4])   # 앞 3행, 앞 4열

In [ ]:
print(df.loc[:4, ["order_id", "date", "category"]])

현재 인덱스가 0, 1, 2... 라서 `.loc[:4]` 는 0~4 라벨을 포함한다. `.iloc[:4]` 는 0~3 위치까지만 포함한다. 이 차이는 시험에도 실무에도 자주 나온다.

> **자주 하는 착각**: `.loc` 의 끝 라벨은 포함, `.iloc` 의 끝 위치는 미포함이다. 슬라이싱 문법이 비슷해서 더 헷갈린다.

---

## 6. 조건 선택과 계산 열 만들기

분석은 결국 필요한 행만 골라 계산하는 일이다. 웹 채널 주문만 골라 보자.

In [ ]:
web_orders = df[df["channel"] == "W"]
print("웹 주문 수:", len(web_orders))
print(web_orders.head())

조건이 여러 개면 괄호와 `&`, `|` 를 쓴다. NumPy boolean indexing 과 같은 규칙이다.

In [ ]:
vip_gadget = df[(df["customer_grade"] == "V") & (df["category"] == "GD")]
print("VIP 기기 주문:", len(vip_gadget))

이제 주문 매출을 계산한다. 할인 전 매출은 `quantity * unit_price`, 할인 후 매출은 여기에 `(1 - discount_rate)` 를 곱한다.

In [ ]:
df["gross_revenue"] = df["quantity"] * df["unit_price"]
df["net_revenue"] = df["gross_revenue"] * (1 - df["discount_rate"])
print(df[["quantity", "unit_price", "discount_rate", "net_revenue"]].head())

pandas 의 열 계산은 행마다 자동으로 적용된다. for 루프를 쓰지 않아도 1000행 전체가 한 번에 계산된다.

---

## 7. 값의 분포 보기: `value_counts`

범주형 데이터는 평균보다 개수 분포가 중요하다.

In [ ]:
print(df["channel"].value_counts())
print(df["category"].value_counts())

비율로 보고 싶으면 `normalize=True` 를 쓴다.

In [ ]:
channel_ratio = df["channel"].value_counts(normalize=True) * 100
print(channel_ratio.round(1))

이 결과로 “웹과 앱이 전체 주문의 대부분을 차지한다” 같은 문장을 쓸 수 있다. 숫자를 표로 보는 것에서 멈추지 말고, 항상 한 문장 해석으로 끝내는 습관을 만든다.

---

## 8. 정렬과 상위 주문 확인

매출이 큰 주문을 보고 싶으면 `sort_values` 를 쓴다.

In [ ]:
top_orders = df.sort_values("net_revenue", ascending=False).head(10)
print(top_orders[["order_id", "category", "quantity", "unit_price", "discount_rate", "net_revenue"]])

상위 주문만 보면 높은 단가 상품이 많은지, 수량이 많은 주문이 많은지 빠르게 판단할 수 있다. 다만 상위 10개만 보고 전체를 결론 내리면 위험하다. 상위 주문은 “후보를 보는 렌즈”이지 전체 요약은 아니다.

In [ ]:
print("총 순매출:", int(df["net_revenue"].sum()))
print("평균 주문금액:", int(df["net_revenue"].mean()))
print("중앙 주문금액:", int(df["net_revenue"].median()))

평균과 중앙값 차이가 크면 고가 주문이 평균을 끌어올렸을 가능성이 있다. 이 해석은 레슨 01의 평균/중앙값 개념과 이어진다.

### 8-1. 정렬 결과를 원본으로 착각하지 않기

`sort_values` 는 기본적으로 정렬된 새 DataFrame 을 돌려준다. 원본 `df` 자체가 바뀌지 않는다.

In [ ]:
sorted_df = df.sort_values("net_revenue", ascending=False)
print("원본 첫 주문:", df.iloc[0]["order_id"])
print("정렬본 첫 주문:", sorted_df.iloc[0]["order_id"])

실무에서는 원본을 보존하는 습관이 중요하다. 특히 필터링과 정렬을 여러 번 하다 보면 “내가 지금 전체 데이터를 보고 있는지, 일부만 보고 있는지” 헷갈리기 쉽다. 변수명을 `web_orders`, `top_orders`, `vip_gadget` 처럼 의미 있게 지으면 실수를 줄일 수 있다.

### 8-2. 숫자 출력 형식

매출은 쉼표가 없으면 읽기 어렵다. f-string 의 `:,.0f` 는 소수 없이 천 단위 쉼표를 붙인다.

In [ ]:
amount = df["net_revenue"].sum()
print(amount)
print(f"{amount:,.0f}원")

데이터 분석 보고서는 계산만 맞는다고 끝나지 않는다. 읽는 사람이 바로 이해할 수 있도록 단위와 형식을 맞춰야 한다.

---

## 9. 현업·대회 활용 사례

### 사례 A. 쇼핑몰 매출 대시보드 첫 화면

커머스 회사에서 매일 아침 보는 기본 지표는 총매출, 주문 수, 평균 주문금액, 채널별 주문 비율이다. 이번 레슨에서 만든 `net_revenue.sum()`, `len(df)`, `net_revenue.mean()`, `value_counts(normalize=True)` 가 그 첫 화면의 계산식이다.

### 사례 B. 캐글 Titanic 데이터 첫 EDA

캐글 Titanic 튜토리얼을 열면 거의 항상 `train.head()`, `train.info()`, `train.describe()`, `train['Sex'].value_counts()` 로 시작한다. 데이터만 승객 정보로 바뀌었을 뿐, 우리가 주문 데이터에서 한 첫 점검과 흐름이 같다.

### 사례 C. 마케팅 캠페인 점검

마케터는 “앱 채널 VIP 고객의 고가 상품 주문이 늘었는가” 같은 질문을 자주 한다. pandas 조건 선택은 이 질문을 코드로 바꾸는 첫 단계다. `df[(grade == 'V') & (channel == 'A')]` 같은 필터가 실제 CRM 분석의 출발점이다.

---

## 10. 따라하기 — 10분 매출 요약 리포트

아래 코드는 오늘 배운 내용을 하나로 묶어 짧은 리포트를 만든다.

In [ ]:
# ✏️ 여기에 코드를 작성하세요



In [ ]:
# ✏️ 여기에 코드를 작성하세요



In [ ]:
# ✏️ 여기에 코드를 작성하세요



이 문장은 숫자, 비교, 결론을 모두 담는다. 최종 미션에서도 같은 방식으로 리포트를 끝낸다.

### 10-1. 직접 해보기

아래 질문은 바로 정답을 보지 말고 먼저 예측한 뒤 코드를 실행한다.

1. 앱 채널 주문만 보면 평균 주문금액이 전체 평균보다 높을까?
2. VIP 고객 주문만 보면 평균 주문금액이 전체 평균보다 높을까?
3. 할인 주문은 무할인 주문보다 평균 주문금액이 낮을까, 높을까?

In [ ]:
# ✏️ 여기에 코드를 작성하세요



<details><summary>해석 예시 보기</summary>

앱 평균이나 VIP 평균이 전체보다 높게 나오면 “앱/VIP 고객의 객단가가 높다”는 가설을 세울 수 있다. 할인 주문 평균이 더 높다고 해서 할인이 무조건 매출을 올렸다고 결론 내리면 안 된다. 고가 상품에 할인이 더 자주 붙었기 때문일 수도 있다. 이런 이유로 다음 레슨 이후에는 여러 조건을 함께 묶어 분석한다.

</details>

---

## 11. pandas 첫 분석 체크리스트

파일을 받을 때마다 아래 순서로 점검한다.

In [ ]:
# ✏️ 여기에 코드를 작성하세요



이 다섯 줄은 너무 기본이라 지루해 보이지만, 실제 오류 대부분은 여기에서 발견된다. 파일을 잘못 읽었거나, 열 이름에 공백이 섞였거나, 숫자 열이 문자열로 읽혔거나, 마지막에 합계 행이 붙은 경우가 모두 첫 점검에서 잡힌다.

---

## 12. 자주 하는 실수

| 증상 | 원인 / 해결 |
|---|---|
| `KeyError: 'revenue'` | 아직 `revenue` 열을 만들지 않았거나 열 이름 오타 |
| `df['a','b']` 가 에러 | 여러 열은 `df[['a', 'b']]` 처럼 리스트로 감싼다 |
| `df.loc[:5]` 와 `df.iloc[:5]` 행 수가 다름 | `.loc` 는 끝 라벨 포함, `.iloc` 는 끝 위치 미포함 |
| 숫자 평균이 이상함 | 가격 열이 문자열로 읽혔는지 `dtypes` 확인 |
| 조건 두 개를 `and` 로 연결 | pandas Series 조건은 `&`, `|` 와 괄호 사용 |
| 원본 `df` 를 계속 덮어씀 | 실험 전 `copy()` 로 작업본을 만든다 |

---

## 13. 다음 단계와 데이터 출처

다음 레슨에서는 CSV 하나를 읽는 수준을 넘어 CSV, Excel, JSON 여러 파일을 불러오고 합친다. 오늘 배운 `head`, `info`, `describe`, 열 선택은 모든 파일 형식에서 그대로 쓰인다.

## 데이터 출처

- `sales_orders.csv` 는 교육용 합성 데이터다(CC0).
- 실제 주문, 고객, 회사 정보가 아니며 개인정보가 없다.
- 생성 의도와 코드표는 `data/README.md` 에 정리되어 있다.